Salary Benchmarking Analysis (Task 1: Data Cleaning & Outliers)

In [1]:
RAW_PATH   = 'SGJobData.csv'          # change path if needed
CLEAN_PATH = 'sg_jobs_salary.parquet'

df_raw = pd.read_csv(RAW_PATH)

print(f'Shape: {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns')
print()
print('Column dtypes:')
print(df_raw.dtypes)
print()
print('Null counts:')
print(df_raw.isnull().sum())

In [3]:
# Always look at a few sample rows before doing anything
df_raw.head(3)

,categories,employmentTypes,metadata_expiryDate,metadata_isPostedOnBehalf,metadata_jobPostId,metadata_newPostingDate,metadata_originalPostingDate,metadata_repostCount,metadata_totalNumberJobApplication,metadata_totalNumberOfView,...,occupationId,positionLevels,postedCompany_name,salary_maximum,salary_minimum,salary_type,status_id,status_jobStatus,title,average_salary
0,"[{""id"":13,""category"":""Environment / Health""},{...",Permanent,2023-05-08,False,MCF-2023-0252866,2023-04-08,2023-03-30,2,5,151,...,NaN,Executive,WORKSTONE PTE. LTD.,2800,2000,Monthly,0,Closed,Food Technologist - Clementi | Entry Level | U...,2400.0
1,"[{""id"":21,""category"":""Information Technology""}]",Permanent,2023-05-08,False,MCF-2023-0273977,2023-04-08,2023-04-08,0,0,55,...,NaN,Executive,TRUST RECRUIT PTE. LTD.,5500,4000,Monthly,0,Closed,"Software Engineer (Fab Support) (Java, CIM, Up...",4750.0
2,"[{""id"":33,""category"":""Repair and Maintenance""}]",Full Time,2023-04-22,False,MCF-2023-0273994,2023-04-08,2023-04-08,0,7,99,...,NaN,Senior Executive,PU TIEN SERVICES PTE. LTD.,4600,3800,Monthly,0,Closed,Senior Technician,4200.0


In [4]:
#Data Quality Check
df.isnull().sum()
(df.isnull().sum() / len(df)) * 100

,salary_minimum,salary_maximum,average_salary
count,1048585.0,1048585.0,1048585.0
mean,3815.0,5724.0,4769.0
std,3172.0,50184.0,25478.0
min,0.0,0.0,0.0
25%,2500.0,3300.0,2900.0
50%,3000.0,4500.0,3800.0
75%,4500.0,6500.0,5500.0
max,350000.0,25330000.0,12666400.0


Key Observations
average_salary has 0 missing values ✅
occupationId has 100% missing ❌ → not useful for analysis
Other columns have minimal missing values (~0.38%) and do not impact salary analysis

In [5]:
# Drop fully missing column
df = df.drop(columns=["occupationId"])

In [6]:
#Salary Missing Values Check
df["average_salary"].isnull().sum()

0

Decision
No imputation required
No rows removed

Reason: Salary is the core variable, and filling values would introduce bias

In [7]:
#Outlier Detection Salary data is typically right-skewed, with a few high-paying roles.We use percentile-based detection:
lower = df["average_salary"].quantile(0.01)
upper = df["average_salary"].quantile(0.99)

lower, upper

['[{"id":13,"category":"Environment / Health"},{"id":25,"category":"Manufacturing"},{"id":36,"category":"Sciences / Laboratory / R&D"}]',
 '[{"id":21,"category":"Information Technology"}]',
 '[{"id":33,"category":"Repair and Maintenance"}]']

In [8]:
#Outlier Handling (Clipping)
df_clean = df.copy()
df_clean["average_salary"] = df_clean["average_salary"].clip(lower, upper)

Unique categories: 43

Top 10 categories:
Admin / Secretarial                 102719
Information Technology              100142
Engineering                          99675
Accounting / Auditing / Taxation     78648
Building and Construction            74014
Customer Service                     64865
F&B                                  59678
Banking and Finance                  46635
Logistics / Supply Chain             44391
Sales / Retail                       37313
Name: category, dtype: int64


Why Clipping?
Preserves all data (no row deletion)
Controls extreme values
Keeps realistic salary variation
Suitable for business benchmarking




#Why not remove top/bottom 5%?

Removing the top and bottom 5% of salary values was not used because it would lead to excessive data loss (10% of the dataset in total). In salary benchmarking, this is not ideal as it may remove valid high-paying and low-paying job roles, resulting in a biased representation of the market.

Fiiling salary value was not required because the average_salary column contains no missing values. Therefore, there was no need to fill or estimate salary values. Additionally, imputing salary values could introduce artificial bias and distort the true salary distribution

In [ ]:
# ── Step 9b: Merge fences back onto the main dataframe ──────────────────────
df = df.merge(
    grp_stats[['positionLevels', 'lower', 'upper']],
    on='positionLevels',
    how='left'
)

# ── Step 9c: Keep only rows within their group's fences ─────────────────────
before = len(df)

df = df[
    (df['average_salary'] >= df['lower']) &
    (df['average_salary'] <= df['upper'])
]

after = len(df)

print(f'Removed : {before - after:,} outlier rows')
print(f'Remaining: {after:,} rows')

# ── Step 9d: Drop the helper columns ────────────────────────────────────────
df = df.drop(columns=['lower', 'upper'])

Removed : 60,282 outlier rows
Remaining: 984,315 rows


In [14]:
# Verify the salary range looks clean now
print('Salary stats after outlier removal:')
print(df.groupby('positionLevels', observed=True)['average_salary'].agg(['min', 'median', 'max']).round(0))

Salary stats after outlier removal:
                      min  median      max
positionLevels                            
Fresh/entry level   500.0  2500.0   5125.0
Non-executive       500.0  2692.0   5450.0
Junior Executive   1000.0  3050.0   5400.0
Executive          1125.0  3750.0   6500.0
Professional        500.0  6000.0  15800.0
Senior Executive    500.0  5000.0  10250.0
Manager             500.0  6000.0  13004.0
Middle Management   500.0  6500.0  17750.0
Senior Management   650.0  9100.0  22125.0


In [15]:
##Before vs After Comparison
plt.boxplot([df["average_salary"], df_clean["average_salary"]])
plt.xticks([1,2], ["Original", "Clipped"])
plt.title("Salary Outliers Before vs After")
plt.show()

Zone stats merged. Sample:


,positionLevels,category,average_salary,Q1,Q3,IQR
0,Executive,Environment / Health,2400.0,3250.0,4500.0,1250.0
1,Executive,Information Technology,4750.0,3750.0,5500.0,1750.0
2,Senior Executive,Repair and Maintenance,4200.0,3500.0,4500.0,1000.0


The boxplot shows a clearer impact of clipping compared to the histogram. While the majority of salary values remain unchanged, extreme values at the upper and lower tails are reduced after applying 1st–99th percentile clipping. This confirms that outliers were effectively controlled without distorting the main salary distribution.

In [10]:
#Summary Statistics Check
df["average_salary"].describe()
df_clean["average_salary"].describe()

count    1.048585e+06
mean     4.622774e+03
std      2.821041e+03
min      9.250000e+01
25%      2.900000e+03
50%      3.800000e+03
75%      5.500000e+03
max      1.666650e+04
Name: average_salary, dtype: float64

In [11]:
# ── Save to Parquet ──────────────────────────────────────────────────────────
df.to_parquet(CLEAN_PATH, index=False)

# Confirm file was written
size_mb = os.path.getsize(CLEAN_PATH) / 1_000_000
print(f'Saved to  : {CLEAN_PATH}')
print(f'File size : {size_mb:.1f} MB')

# Quick reload test — make sure it reads back correctly
df_test = pd.read_parquet(CLEAN_PATH)
print(f'Reload check: {len(df_test):,} rows × {df_test.shape[1]} columns  ✓')
print()
print('Column dtypes after reload:')
print(df_test.dtypes)

Saved to  : sg_jobs_salary.parquet
File size : 41.1 MB
Reload check: 984,315 rows × 24 columns  ✓

Column dtypes after reload:
employmentTypes                               object
metadata_isPostedOnBehalf                       bool
metadata_repostCount                           int64
metadata_totalNumberJobApplication             int64
metadata_totalNumberOfView                     int64
minimumYearsExperience                         int64
numberOfVacancies                              int64
positionLevels                              category
postedCompany_name                            object
salary_maximum                                 int64
salary_minimum                                 int64
status_jobStatus                              object
title                                         object
average_salary                               float64
category                                      object
exp_band                                    category
Q1                       

✔ data is skewed
✔ outliers exist at extremes
✔ clipping preserves dataset integrity
✔ cleaned data is ready for benchmarking

#The maximum salary value decreased significantly after clipping (from 12,666,400 to 16,666.5), confirming the presence of extreme outliers in the original dataset. These values were likely data anomalies or rare cases that could distort salary benchmarking. Percentile-based clipping effectively controlled these extremes while preserving the overall dataset structure.